# Reinforcement Learning — The Math, In Code
This notebook walks through every equation from the RL slide-deck series (Episodes 1–4) and computes each one with real numbers, so you can see exactly what each formula does rather than just reading it.

Sections:
1. Setup (Groq API, for an optional "ask a question" helper at the end)
2. Expectation — the weighted average
3. Return & discounting
4. Value function V(s) via Monte Carlo estimation
5. Bellman equation — exact solve via linear algebra
6. Q-Learning update rule
7. SARSA vs. Q-Learning
8. Gradient descent, from scratch
9. DQN loss (toy linear function approximator)
10. Double DQN — demonstrating and fixing maximization bias
11. Dueling DQN — the V + A decomposition
12. Prioritized Experience Replay — priorities & sampling
13. Multi-step returns
14. Distributional RL — the categorical projection
15. Noisy Networks — learnable exploration noise
16. Bonus: ask an LLM (Groq) to explain any concept in your own words


## 1. Setup
This cell sets your Groq API key as an environment variable (used only in the bonus section at the end — everything else in this notebook is plain NumPy, no API calls needed).

⚠️ **Note on the key below:** treat any real API key as a secret. It's fine to hardcode it while you're experimenting alone, but avoid committing a notebook with a real key to any shared or public repository — swap to `getpass()` (commented out below) if you ever share this file.

In [ ]:
import os
from getpass import getpass

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = "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"

# If you'd rather not hardcode a key, comment out the block above and uncomment this instead:
# if "GROQ_API_KEY" not in os.environ:
#     os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

import numpy as np
np.set_printoptions(precision=4, suppress=True)
print("Setup complete. NumPy version:", np.__version__)

## 2. Expectation — The Weighted Average
𝔼[X] = Σₓ P(x)·x — multiply every outcome by how likely it is, then add them up.

Same spinner example as the slides: 50% chance of 10 points, 30% chance of 4 points, 20% chance of 0 points.

In [ ]:
outcomes = np.array([10, 4, 0])
probabilities = np.array([0.5, 0.3, 0.2])

assert np.isclose(probabilities.sum(), 1.0), "probabilities must sum to 1"

expectation = np.sum(outcomes * probabilities)
print(f"E[X] = {outcomes} weighted by {probabilities}")
print(f"E[X] = {expectation}")

# Verify with simulation: sample many times and average
rng = np.random.default_rng(seed=0)
samples = rng.choice(outcomes, size=200_000, p=probabilities)
print(f"Simulated average over 200,000 draws: {samples.mean():.4f}  (should be close to {expectation})")

## 3. Return & Discounting
G_t = r₁ + γr₂ + γ²r₃ + ⋯

We'll compute the return for a fixed reward sequence under a few different discount factors, and see how γ changes how much "the future" counts.

In [ ]:
def compute_return(rewards, gamma):
    """G_t = sum of gamma^k * r_{t+k+1} for k = 0, 1, 2, ..."""
    discounts = gamma ** np.arange(len(rewards))
    return np.sum(discounts * rewards)

rewards = np.array([1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0])  # constant reward of 1 for 10 steps

for gamma in [0.0, 0.5, 0.9, 0.99, 1.0]:
    G = compute_return(rewards, gamma)
    print(f"gamma = {gamma:>4}:  G_t = {G:.4f}")

print()
print("As gamma -> 1, the return approaches the un-discounted sum (10.0 here).")
print("The geometric-series formula for infinite constant reward r forever: G = r / (1 - gamma)")
r = 1.0
for gamma in [0.5, 0.9, 0.99]:
    print(f"  gamma={gamma}: closed-form infinite return = {r/(1-gamma):.2f}")

## 4. Value Function V(s) via Monte Carlo Estimation
V(s) = 𝔼[G_t | s_t = s] — the average return, sampled by actually simulating many trajectories from state s.

We'll use a tiny toy environment: from state s, each step gives a random reward (Normal, mean=1, std=2), and the episode ends after a random number of steps (geometric distribution). We estimate V(s) purely by simulation — no formula, just averaging.

In [ ]:
def simulate_episode(gamma, rng, continue_prob=0.85, reward_mean=1.0, reward_std=2.0, max_steps=200):
    """Simulate one episode from state s: random reward each step, random episode length."""
    total_return = 0.0
    discount = 1.0
    for step in range(max_steps):
        reward = rng.normal(reward_mean, reward_std)
        total_return += discount * reward
        discount *= gamma
        if rng.random() > continue_prob:  # episode ends
            break
    return total_return

rng = np.random.default_rng(seed=42)
gamma = 0.9
n_episodes = 20_000

returns = np.array([simulate_episode(gamma, rng) for _ in range(n_episodes)])
V_estimate = returns.mean()
V_std_error = returns.std() / np.sqrt(n_episodes)

print(f"V(s) estimated from {n_episodes} simulated episodes: {V_estimate:.4f} (+/- {1.96*V_std_error:.4f} at 95% confidence)")
print(f"Return variance across episodes: {returns.var():.4f}  <- this is exactly why Monte Carlo can be noisy")

## 5. Bellman Equation — Exact Solve via Linear Algebra
For a small, fully-known MDP we can solve the Bellman expectation equation exactly:

**V = R + γPV  ⟹  V = (I − γP)⁻¹ R**

Below: a tiny 3-state MDP (with a fixed policy already baked into P and R) solved exactly, then cross-checked against value iteration (repeatedly applying the Bellman update) to confirm they agree.

In [ ]:
gamma = 0.9

# 3 states. P[i, j] = probability of moving from state i to state j (under the fixed policy).
P = np.array([
    [0.5, 0.3, 0.2],
    [0.1, 0.6, 0.3],
    [0.2, 0.2, 0.6],
])
# R[i] = expected immediate reward for being in state i (under the fixed policy).
R = np.array([1.0, 0.0, 2.0])

assert np.allclose(P.sum(axis=1), 1.0), "each row of P must sum to 1 (it's a probability distribution)"

# --- Exact solve: V = (I - gamma*P)^-1 R ---
I = np.eye(3)
V_exact = np.linalg.solve(I - gamma * P, R)
print("Exact solve,      V =", V_exact)

# --- Cross-check: value iteration (repeatedly apply the Bellman backup) ---
V_iter = np.zeros(3)
for i in range(1000):
    V_iter = R + gamma * P @ V_iter

print("Value iteration,  V =", V_iter)
print("Difference:", np.abs(V_exact - V_iter).max(), " <- should be ~0, confirming both methods agree")

## 6. Q-Learning Update Rule
Q(s,a) ← Q(s,a) + α [ r + γ·maxₐ' Q(s',a') − Q(s,a) ]

Reproducing the exact worked example from the slides: Q=3.0, r=2, γ=0.9, max Q(s')=5.0, α=0.1.

In [ ]:
def q_learning_update(Q_sa, reward, gamma, max_next_Q, alpha):
    td_target = reward + gamma * max_next_Q
    td_error = td_target - Q_sa
    new_Q = Q_sa + alpha * td_error
    return new_Q, td_target, td_error

Q_sa = 3.0
reward = 2
gamma = 0.9
max_next_Q = 5.0
alpha = 0.1

new_Q, target, delta = q_learning_update(Q_sa, reward, gamma, max_next_Q, alpha)
print(f"TD target = r + gamma*max_Q(s') = {reward} + {gamma}*{max_next_Q} = {target}")
print(f"TD error (delta)               = target - Q(s,a) = {target} - {Q_sa} = {delta}")
print(f"Updated Q(s,a)                 = {Q_sa} + {alpha}*{delta} = {new_Q}")

# Now run it repeatedly and watch Q converge toward the target
print("\nWatching Q(s,a) converge over repeated updates with a fixed target:")
Q_sa = 3.0
for step in range(15):
    Q_sa, target, delta = q_learning_update(Q_sa, reward, gamma, max_next_Q, alpha)
    print(f"  step {step+1:>2}: Q = {Q_sa:.4f}")

## 7. SARSA vs. Q-Learning
Both use the same update shape; the only difference is what stands in for "the next action's value":
- **Q-learning (off-policy):** uses `max_a' Q(s',a')` — the best possible next action.
- **SARSA (on-policy):** uses `Q(s',a')` for the action the agent *actually* takes next (including exploratory moves).

We'll simulate a simplified "cliff-walk"-style bandit choice to see the practical difference: SARSA ends up more cautious because it accounts for the real chance of an exploratory misstep.

In [ ]:
rng = np.random.default_rng(1)

# Two actions from a state near a cliff: "risky" (great if it works, terrible if exploration causes a slip)
# and "safe" (always mediocre but fine).
def sample_transition(action, rng):
    if action == "risky":
        # 90% of the time: big reward. 10% of the time (a "slip"): big penalty.
        return 10.0 if rng.random() < 0.9 else -100.0
    else:
        return 1.0  # safe action: small, reliable reward

epsilon = 0.1  # exploration rate used by BOTH methods' behavior policy

def run(method, n_episodes=20000):
    Q = {"risky": 0.0, "safe": 0.0}
    alpha, gamma = 0.1, 0.9
    for _ in range(n_episodes):
        # epsilon-greedy action selection (this is what's ACTUALLY taken)
        action = rng.choice(["risky", "safe"]) if rng.random() < epsilon else max(Q, key=Q.get)
        reward = sample_transition(action, rng)

        if method == "q_learning":
            target = reward + gamma * max(Q.values())  # off-policy: assumes best next action
        elif method == "sarsa":
            next_action = rng.choice(["risky", "safe"]) if rng.random() < epsilon else max(Q, key=Q.get)
            target = reward + gamma * Q[next_action]   # on-policy: uses the actual next action

        Q[action] += alpha * (target - Q[action])
    return Q

Q_qlearning = run("q_learning")
Q_sarsa = run("sarsa")

print("Learned Q-values:")
print(f"  Q-learning: {Q_qlearning}   -> prefers: {max(Q_qlearning, key=Q_qlearning.get)}")
print(f"  SARSA:      {Q_sarsa}   -> prefers: {max(Q_sarsa, key=Q_sarsa.get)}")
print("\nQ-learning tends to rate 'risky' more optimistically (it assumes the best case, no slips),")
print("while SARSA's value for 'risky' factors in the real, occasional exploratory slip -> more caution.")

## 8. Gradient Descent, From Scratch
θ ← θ − α·∇_θ L(θ)

A minimal 1D example: minimize L(θ) = (θ − 4)² by hand-computing the gradient at each step (no autodiff library needed to see the idea).

In [ ]:
def loss(theta):
    return (theta - 4) ** 2

def grad(theta):
    return 2 * (theta - 4)   # d/dtheta (theta-4)^2 = 2(theta-4)

theta = 0.0
alpha = 0.1
history = [theta]

for step in range(30):
    g = grad(theta)
    theta = theta - alpha * g
    history.append(theta)

print("theta over the first 10 steps:", np.round(history[:10], 4))
print("theta after 30 steps:", round(theta, 6), " (true minimum is at theta=4)")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
thetas = np.linspace(-2, 8, 200)
ax[0].plot(thetas, loss(thetas), color="steelblue")
ax[0].scatter(history, [loss(t) for t in history], color="orange", s=15, zorder=3)
ax[0].set_title("Descending the loss surface")
ax[0].set_xlabel("theta"); ax[0].set_ylabel("L(theta)")

ax[1].plot(history, marker="o", markersize=3, color="steelblue")
ax[1].axhline(4, color="gray", linestyle="--", label="true minimum (theta=4)")
ax[1].set_title("theta approaching the minimum")
ax[1].set_xlabel("step"); ax[1].set_ylabel("theta"); ax[1].legend()
plt.tight_layout()
plt.savefig("gradient_descent.png", dpi=110)
plt.show()
print("Saved plot to gradient_descent.png")

## 9. DQN Loss (Toy Linear Function Approximator)
L(θ) = 𝔼[(r + γ·maxₐ' Q(s',a';θ⁻) − Q(s,a;θ))²]

Using the exact same toy example from the slides: Q(s,a;θ) = θ·x, with x=2, θ=1.5.

In [ ]:
theta = 1.5
x = 2.0
r = 2.0
gamma = 0.9
alpha = 0.1
max_next_Q_target_net = 5.0  # maxₐ' Q(s',a';theta_minus), from the FROZEN target network

Q_pred = theta * x
target = r + gamma * max_next_Q_target_net
delta = target - Q_pred
loss_value = delta ** 2

# Gradient of L(theta) = (target - theta*x)^2 w.r.t. theta:
#   dL/dtheta = 2*(target - theta*x)*(-x) = -2*delta*x
grad_theta = -2 * delta * x
theta_new = theta - alpha * grad_theta
Q_new = theta_new * x

print(f"Prediction Q(s,a;theta)  = theta*x = {theta}*{x} = {Q_pred}")
print(f"Target                   = r + gamma*maxQ(s';theta-) = {r} + {gamma}*{max_next_Q_target_net} = {target}")
print(f"TD error (delta)         = {delta}")
print(f"Loss = delta^2           = {loss_value}")
print(f"Gradient dL/dtheta       = {grad_theta}")
print(f"Updated theta            = {theta} - {alpha}*({grad_theta}) = {theta_new}")
print(f"New Q(s,a;theta)         = {theta_new}*{x} = {Q_new}   (closer to the target {target}, as expected)")

## 10. Double DQN — Demonstrating and Fixing Maximization Bias
Claim from the slides: taking the max over several *noisy* estimates of the same true value systematically overestimates that true value.

We'll verify this with simulation, then show how Double DQN's "decouple selection from evaluation" trick reduces (though doesn't eliminate) the bias.

In [ ]:
rng = np.random.default_rng(7)
true_value = 5.0
n_actions = 4
noise_std = 0.5
n_trials = 50_000

# --- Plain DQN-style max: one noisy estimate per action, pick the max ---
plain_max_estimates = []
for _ in range(n_trials):
    noisy_Q = rng.normal(true_value, noise_std, size=n_actions)
    plain_max_estimates.append(noisy_Q.max())
plain_max_estimates = np.array(plain_max_estimates)

print(f"True value: {true_value}")
print(f"Average of plain max-of-noisy-estimates over {n_trials} trials: {plain_max_estimates.mean():.4f}")
print(f"  -> overestimation bias: {plain_max_estimates.mean() - true_value:.4f}\n")

# --- Double-DQN style: TWO independent noisy estimators.
# Estimator A picks which action looks best; estimator B scores that chosen action.
double_estimates = []
for _ in range(n_trials):
    noisy_Q_A = rng.normal(true_value, noise_std, size=n_actions)  # "online network"
    noisy_Q_B = rng.normal(true_value, noise_std, size=n_actions)  # "target network"
    chosen_action = noisy_Q_A.argmax()          # selection uses A
    double_estimates.append(noisy_Q_B[chosen_action])  # evaluation uses B
double_estimates = np.array(double_estimates)

print(f"Average of Double-DQN-style decoupled estimate over {n_trials} trials: {double_estimates.mean():.4f}")
print(f"  -> overestimation bias: {double_estimates.mean() - true_value:.4f}")
print("\nNotice the Double-DQN-style bias is much closer to zero than the plain max's bias.")

## 11. Dueling DQN — The V + A Decomposition
Q(s,a) = V(s) + [A(s,a) − mean_a' A(s,a')]

We'll also demonstrate the *identifiability problem* the mean-subtraction fixes: without it, infinitely many (V, A) pairs give the same Q.

In [ ]:
# A "calm" state: action barely matters (advantages all close to 0)
V_calm = 8.0
A_calm = np.array([0.1, -0.1, 0.05, -0.05])
A_calm_centered = A_calm - A_calm.mean()          # should already be ~centered
Q_calm = V_calm + A_calm_centered
print("Calm state:")
print(f"  V(s) = {V_calm}, A(s,a) = {A_calm}")
print(f"  Q(s,a) = {Q_calm}\n")

# A "critical" state: action matters a lot
V_crit = 2.0
A_crit = np.array([6.0, -3.0, 0.5, -2.5])
A_crit_centered = A_crit - A_crit.mean()
Q_crit = V_crit + A_crit_centered
print("Critical state:")
print(f"  V(s) = {V_crit}, A(s,a) = {A_crit}")
print(f"  mean-centered A = {A_crit_centered}  (mean is now exactly 0)")
print(f"  Q(s,a) = {Q_crit}\n")

# --- The identifiability problem, demonstrated ---
print("Identifiability problem: shift V by +10 and A by -10, get the EXACT SAME Q:")
V_shifted = V_crit + 10
A_shifted = A_crit - 10
Q_shifted = V_shifted + (A_shifted - A_shifted.mean())
print(f"  V={V_shifted}, A={A_shifted}  ->  Q = {Q_shifted}")
print(f"  Same as before? {np.allclose(Q_shifted, Q_crit)}")
print("This is exactly why V and A are meaningless individually without the mean-subtraction constraint.")

## 12. Prioritized Experience Replay — Priorities & Sampling
pᵢ = |δᵢ| + ε &nbsp;&nbsp; P(i) = pᵢ^α / Σₖ pₖ^α

We'll compute sampling probabilities for a small buffer of transitions with different TD errors, and show how much more often the "big surprise" transition gets sampled compared to plain uniform sampling.

In [ ]:
td_errors = np.array([0.05, 0.03, 0.08, 0.02, 4.0, 0.04, 0.06, 0.01, 0.09, 0.02])  # one big surprise (index 4)
epsilon = 0.01
alpha_priority = 1.0  # how strongly priority matters (alpha=0 -> uniform, alpha=1 -> fully proportional)

priorities = np.abs(td_errors) + epsilon
sampling_probs = priorities ** alpha_priority / np.sum(priorities ** alpha_priority)
uniform_probs = np.ones_like(td_errors) / len(td_errors)

print("Transition:      ", np.arange(len(td_errors)))
print("TD error:         ", td_errors)
print("Uniform P(i):     ", np.round(uniform_probs, 4))
print("Prioritized P(i): ", np.round(sampling_probs, 4))
print()
ratio = sampling_probs[4] / uniform_probs[4]
print(f"The big-surprise transition (index 4) is sampled {ratio:.1f}x more often under prioritized replay than uniform.")

# --- Importance-sampling correction weights ---
N = len(td_errors)
beta = 0.5  # partially corrected; beta -> 1 over training in real implementations
is_weights = (1 / (N * sampling_probs)) ** beta
is_weights_normalized = is_weights / is_weights.max()  # normalize for training stability
print("\nImportance-sampling weights (normalized):", np.round(is_weights_normalized, 4))
print("Notice the oft-sampled transition (index 4) gets the SMALLEST weight, correcting for how often it's seen.")

## 13. Multi-Step Returns
Gₜ⁽ⁿ⁾ = r₁ + γr₂ + ⋯ + γⁿ⁻¹rₙ + γⁿ·maxₐ Q(s_{t+n}, a)

We'll compute this for several values of n on the same reward sequence, to see the bias/variance dial in action.

In [ ]:
rewards = np.array([1.0, 0.0, 2.0, 0.5, 1.5])  # 5 real rewards observed
gamma = 0.9
bootstrap_value = 6.0  # Q(s_{t+n}, a) at whatever step we stop bootstrapping

def n_step_return(rewards, gamma, n, bootstrap_value):
    n = min(n, len(rewards))
    discounts = gamma ** np.arange(n)
    real_part = np.sum(discounts * rewards[:n])
    bootstrap_part = (gamma ** n) * bootstrap_value
    return real_part + bootstrap_part

for n in [1, 2, 3, 5]:
    G_n = n_step_return(rewards, gamma, n, bootstrap_value)
    print(f"n={n}:  G_t^(n) = {G_n:.4f}   (n=1 is plain TD; n=len(rewards) with no bootstrap left is like Monte Carlo)")

## 14. Distributional RL — The Categorical Projection
Instead of one Q-value, learn a full distribution over a fixed set of "atoms" (possible return values), then re-project a shifted/scaled distribution back onto those same fixed atoms.

In [ ]:
# Fixed atoms (support) for the return distribution
atoms = np.array([-10, -5, 0, 5, 10, 15, 20])
n_atoms = len(atoms)

# A distribution over those atoms (must sum to 1)
probs = np.array([0.02, 0.03, 0.10, 0.30, 0.35, 0.15, 0.05])
assert np.isclose(probs.sum(), 1.0)

Q_from_distribution = np.sum(atoms * probs)
print("Original distribution over atoms:", dict(zip(atoms, probs)))
print(f"Q(s,a) recovered as the mean of the distribution: {Q_from_distribution:.4f}\n")

# Apply the distributional Bellman update: shift by reward r, scale by gamma
r = 3.0
gamma = 0.9
shifted_scaled_atoms = r + gamma * atoms
print("After applying r + gamma*z to every atom, the new support no longer lines up with the original atoms:")
print(np.round(shifted_scaled_atoms, 2))

# --- Project back onto the original fixed atoms ---
def project_distribution(shifted_atoms, probs, original_atoms):
    v_min, v_max = original_atoms[0], original_atoms[-1]
    delta_z = original_atoms[1] - original_atoms[0]
    projected = np.zeros_like(original_atoms, dtype=float)

    for z, p in zip(shifted_atoms, probs):
        z_clamped = np.clip(z, v_min, v_max)
        b = (z_clamped - v_min) / delta_z  # fractional position among original atoms
        lower = int(np.floor(b))
        upper = min(lower + 1, len(original_atoms) - 1)
        weight_upper = b - lower
        weight_lower = 1 - weight_upper
        projected[lower] += p * weight_lower
        projected[upper] += p * weight_upper
    return projected

projected_probs = project_distribution(shifted_scaled_atoms, probs, atoms)
print("\nProjected back onto the original atoms:", np.round(projected_probs, 4))
print("Total probability preserved:", round(projected_probs.sum(), 6), " (should still be 1.0)")
print(f"New Q-value from projected distribution: {np.sum(atoms * projected_probs):.4f}")

## 15. Noisy Networks — Learnable Exploration Noise
y = (μ_w + σ_w ⊙ ε_w)x + (μ_b + σ_b ⊙ ε_b)

We'll simulate a single noisy linear layer, sampling fresh noise every forward pass, and show how shrinking σ (as if the network became more confident) reduces the spread of outputs — replacing a fixed epsilon with a *tunable, learnable* amount of randomness.

In [ ]:
rng = np.random.default_rng(3)

def noisy_linear_forward(x, mu_w, sigma_w, mu_b, sigma_b, rng):
    eps_w = rng.normal(size=mu_w.shape)
    eps_b = rng.normal(size=mu_b.shape)
    w = mu_w + sigma_w * eps_w
    b = mu_b + sigma_b * eps_b
    return w @ x + b

x = np.array([1.0, 0.5])
mu_w = np.array([2.0, -1.0])
mu_b = np.array([0.5])

print("Outputs across 8 forward passes with HIGH noise (sigma=1.0, i.e. still uncertain):")
outputs_high_noise = [noisy_linear_forward(x, mu_w, np.full(2, 1.0), mu_b, np.full(1, 1.0), rng)[0] for _ in range(8)]
print(np.round(outputs_high_noise, 3), " std =", round(np.std(outputs_high_noise), 3))

print("\nOutputs across 8 forward passes with LOW noise (sigma=0.05, i.e. network has learned to be confident):")
outputs_low_noise = [noisy_linear_forward(x, mu_w, np.full(2, 0.05), mu_b, np.full(1, 0.05), rng)[0] for _ in range(8)]
print(np.round(outputs_low_noise, 3), " std =", round(np.std(outputs_low_noise), 3))

print("\nThis is exactly the mechanism: gradient descent can shrink sigma in states/parameters")
print("the network is confident about, automatically tapering exploration without a hand-set schedule.")

## 16. Bonus: Ask An LLM To Explain Any Concept
Uses the Groq API (OpenAI-compatible chat completion) to answer follow-up questions about anything in this notebook, in case you want a different phrasing or a deeper dive on one specific piece.

Requires: `pip install groq`

In [ ]:
# pip install groq   (uncomment the line below if you haven't installed it yet)
# !pip install groq -q

from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

def ask_about_rl(question, model="llama-3.3-70b-versatile"):
    """Ask a Groq-hosted LLM to explain an RL math concept in plain language."""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": (
                "You are a patient reinforcement learning tutor. Explain concepts in plain, "
                "concrete language, using small numeric examples where possible. Assume the "
                "learner already knows: expectation, the Bellman equation, Q-learning, DQN, "
                "Double DQN, Dueling DQN, and Rainbow DQN's components."
            )},
            {"role": "user", "content": question},
        ],
        temperature=0.3,
    )
    return response.choices[0].message.content

# Example usage:
answer = ask_about_rl("Why does subtracting the mean advantage fix the identifiability problem in Dueling DQN?")
print(answer)

---
### Where to go from here
- Swap the toy environments in sections 4, 6, and 7 for a real Gymnasium environment (`pip install gymnasium`) to see these exact same equations at work on `CliffWalking-v0` or `FrozenLake-v1`.
- Replace the toy linear function approximator in section 9 with a small PyTorch network to build a real DQN.
- Use the `ask_about_rl()` helper from section 16 to quiz yourself — try asking it to check a calculation you did by hand.
